# 03_07_Vertical_Exaggeration_Of_Elevation

## 1. Introduction

Real-world landscapes often have large horizontal extents compared to relatively small variations in elevation.

For Waterloo Region, the study area spans approximately:

- ~55 km east–west
- ~47 km north–south
- ~191 m of total elevation change

Because vertical relief is small relative to horizontal distance, a true-scale 3D terrain model often appears visually flat.

Vertical exaggeration is a visualization technique that scales the elevation (Z axis) while leaving horizontal dimensions unchanged. This improves the visibility of terrain features such as valleys, ridges, and drainage patterns.

In this notebook:

- Horizontal axes remain in kilometers (as established in previous notebooks)
- Elevation values are converted from meters to kilometers (for scale consistency)
- A multiplicative exaggeration factor is applied to the Z axis only
  
Important interpretation note:

***The color scale and Z-axis values shown in the plot correspond to the scaled elevation values used for visualization.***

This means:

- The legend reflects exaggerated elevation values, not raw meters
- The plot is mathematically correct for the transformed dataset
- It should not be interpreted as raw elevation magnitude

Ultimately this means that the legend is going display elevation values in km and scaled by the scaling factor, not the actual elevation values. Plotly controls how these values are rendered, and the scaling is not separated into a “visual-only legend layer”.

Another important point is that this notebook intentionally re-computes the terrain model from clipped_dem and it does not depend on the previous notebook. As the previous notebook explains the visualization we created in the last part of that notebook is a visualization only, and not a reusable dataset. To keep things concise, I am not going to re-explain all of the operations involved in creating the visualization from the dataset again in this notebook, but you can look at the previous notebook for clarification if needed. I have left comments in the code which outline each step, without the full explanations. 

## 2. Imports

Import core geospatial and visualization libraries used in this notebook:

In [5]:
import rasterio
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets

## 3. Load Data

The clipped DEM contains NoData values outside the valid elevation surface. These values are converted to NaN so that statistical calculations and visualizations ignore missing data.

In [6]:
# Load clipped DEM (recomputed locally for independence from previous notebooks)
with rasterio.open("../../data/Elevation/clipped_dem.tif") as src:
    z = src.read(1)
    bounds = src.bounds
    nodata = src.nodata

# Convert NoData values to NaN for safe computation (ignore missing raster areas)
z = np.where(z == nodata, np.nan, z)

## 4. Vertical Exaggeration Terrain Function

In [13]:
def plot_terrain(vertical_exaggeration=1):

    # --- Recompute spatial reference (do not rely on external state) ---
    with rasterio.open("../../data/clipped_dem.tif") as src:
        z = src.read(1)
        bounds = src.bounds
        nodata = src.nodata

    # Handle missing data
    z = np.where(z == nodata, np.nan, z)

    rows, cols = z.shape

    # --- Convert geographic extent to approximate kilometers ---
    mean_lat = (bounds.top + bounds.bottom) / 2
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(mean_lat))

    width_km = (bounds.right - bounds.left) * km_per_deg_lon
    height_km = (bounds.top - bounds.bottom) * km_per_deg_lat

    # --- Build spatial grid (consistent with previous notebooks) ---
    x = np.linspace(0, width_km, cols)
    y = np.linspace(0, height_km, rows)

    # --- Downsample for performance (visual only) ---
    step = 4
    z_small = z[::step, ::step]
    x_small = x[::step]
    y_small = y[::step]

    # --- Correct raster orientation (Plotly vs raster origin mismatch) ---
    z_small = np.flipud(z_small)

    # --- Convert elevation to km and apply vertical exaggeration ---
    z_small = (z_small / 1000.0) * vertical_exaggeration

    fig = go.Figure(
        data=[
            go.Surface(
                x=x_small,
                y=y_small,
                z=z_small,
                colorscale="Emrld"
            )
        ]
    )

    fig.update_layout(
        title=f"Waterloo Region Terrain (Exaggeration: {vertical_exaggeration}×)",

        scene=dict(
            xaxis_title="Distance (km)",
            yaxis_title="Distance (km)",
            zaxis_title="Elevation (km)",

            aspectmode="manual",
            aspectratio=dict(
                x=width_km,
                y=height_km,
                z=(np.nanmax(z_small) - np.nanmin(z_small))
            )
        )
    )

    fig.show()

## 5. Interactive Slider

In [14]:
slider = widgets.IntSlider(
    value=20,
    min=1,
    max=30,
    step=1,
    description="Vertical Exaggeration:"
)

widgets.interact(plot_terrain, vertical_exaggeration=slider)

interactive(children=(IntSlider(value=20, description='Vertical Exaggeration:', max=30, min=1), Output()), _do…

<function __main__.plot_terrain(vertical_exaggeration=1)>